In [2]:
# -------------------------------
# Step 1: Import required libraries
# -------------------------------
import gymnasium as gym  # Modern replacement for 'gym'
import numpy as np

# -------------------------------
# Step 2: Create the environment
# -------------------------------
# We use is_slippery=True to make transitions probabilistic (stochastic),
# which is perfect for demonstrating Monte Carlo's empirical learning.
env = gym.make("FrozenLake-v1", is_slippery=True)

# -------------------------------
# Step 3: Define a Random Policy
# -------------------------------
# MC prediction evaluates a given policy. We'll evaluate a uniform random policy
# where each of the 4 actions has an equal (25%) chance of selection.
num_states = env.observation_space.n
num_actions = env.action_space.n


def random_policy(state):
    """Returns a completely random action for any given state."""
    return np.random.choice(num_actions)


# -------------------------------
# Step 4: Generate an Episode
# -------------------------------
def generate_episode(env, policy):
    """Runs a single full episode from start to finish using the policy.

    Returns a list of tuples: (state, action, reward)
    """
    episode = []
    state, info = env.reset()  # Reset env to get initial state
    done = False

    while not done:
        action = policy(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # Append step experience
        episode.append((state, action, reward))
        state = next_state

    return episode


# -------------------------------
# Step 5: First-Visit MC Prediction
# -------------------------------
def monte_carlo_prediction(env, policy, num_episodes, gamma=0.99):
    """Calculates the state-value function V(s) using First-Visit MC."""
    # Initialize value table to zeros
    V = np.zeros(num_states)

    # Track returns accumulated for each state: {state: [return1, return2, ...]}
    returns = {state: [] for state in range(num_states)}

    for ep in range(num_episodes):
        # 1. Generate a complete episode
        episode = generate_episode(env, policy)

        # 2. Track total returns and first visits
        G = 0
        visited_states = set()

        # Step backwards through the episode to calculate returns smoothly
        for t in range(len(episode) - 1, -1, -1):
            state, action, reward = episode[t]

            # Accumulate discounted target reward: G_t = R_{t+1} + gamma * G_{t+1}
            G = reward + gamma * G

            # First-visit check: check if state was visited earlier in this episode
            # If we don't look backwards or manage a set, we might count multi-visits.
            # We check the prefix of the episode up to step t:
            state_history = [step[0] for step in episode[:t]]

            if state not in state_history:
                returns[state].append(G)
                # Update value function as the empirical average of returns
                V[state] = np.mean(returns[state])

    return V


# -------------------------------
# Step 6: Run the MC Prediction
# -------------------------------
print("Running First-Visit Monte Carlo Prediction...")
total_episodes = 10000
discount_factor = 0.95

# Estimate values
estimated_value_table = monte_carlo_prediction(
    env, random_policy, num_episodes=total_episodes, gamma=discount_factor
)

# -------------------------------
# Step 7: Print the Output
# -------------------------------
print(f"\nEstimated Value Function V(s) after {total_episodes} episodes:")
# Reshape the 16 states into a 4x4 grid corresponding to the FrozenLake layout
print(np.round(estimated_value_table.reshape(4, 4), 5))

# Close the environment cleanly
env.close()


Running First-Visit Monte Carlo Prediction...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Estimated Value Function V(s) after 10000 episodes:
[[0.00844 0.0077  0.01597 0.00703]
 [0.011   0.      0.03641 0.     ]
 [0.02722 0.07473 0.13026 0.     ]
 [0.      0.16871 0.43573 0.     ]]
